# V7_0_N02 — Build the Evidence Contract

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. Illustrative data do not constitute official statistics or operational authorization.

## Learning outcomes
Define entities, identifiers, measures, classifications, reference periods, spatial grain, provenance, lawful access, linkage rules, and retention before analysis.

## 1. Evidence is a contract
A table is not self-explanatory. The same column name can represent different concepts across agencies. A usable source contract states what each record means, who created it, when it applies, and which transformations are permitted.

In [1]:
from dataclasses import dataclass,asdict
import pandas as pd, json
@dataclass
class SourceContract:
 source_id:str; sector:str; authority:str; entity:str; time_grain:str; spatial_grain:str; key:str; update_cycle:str; lawful_basis:str; retention:str; quality_owner:str
contracts=[
 SourceContract('AGR_FRAME','Agriculture','National Statistical Office','agricultural holding','annual','enumeration area','holding_id','annual','statistical mandate','controlled schedule','Census methods unit'),
 SourceContract('HMIS_M','Health','Ministry of Health','facility service aggregate','monthly','health facility','facility_month','monthly','health information mandate','approved aggregate retention','HMIS unit'),
 SourceContract('EMIS_T','Education','Ministry of Education','school-term aggregate','term','school','school_term','term','education statistics mandate','approved aggregate retention','EMIS unit'),
 SourceContract('SETTLE_G','Habitat','Planning authority','settlement grid cell','quarter','1 km grid','grid_period','quarter','planning mandate','review every 24 months','Geospatial unit')]
print(pd.DataFrame([asdict(x) for x in contracts]).to_string(index=False))

source_id      sector                   authority                     entity time_grain    spatial_grain            key update_cycle                 lawful_basis                    retention       quality_owner
AGR_FRAME Agriculture National Statistical Office       agricultural holding     annual enumeration area     holding_id       annual          statistical mandate          controlled schedule Census methods unit
   HMIS_M      Health          Ministry of Health facility service aggregate    monthly  health facility facility_month      monthly   health information mandate approved aggregate retention           HMIS unit
   EMIS_T   Education       Ministry of Education      school-term aggregate       term           school    school_term         term education statistics mandate approved aggregate retention           EMIS unit
 SETTLE_G     Habitat          Planning authority       settlement grid cell    quarter        1 km grid    grid_period      quarter             planning ma

## 2. Validate identifiers and grain
A valid key must be unique at the declared entity-time-space grain. Failure means aggregation, duplication, or linkage may produce false counts.

In [2]:
records=pd.DataFrame({'source_id':['AGR_FRAME','HMIS_M','EMIS_T','SETTLE_G'],'declared_key':['holding_id','facility_month','school_term','grid_period'],'unique_key':[True,True,True,True],'reference_period_present':[True,True,True,True],'geography_versioned':[True,False,True,True]})
records['contract_pass']=records[['unique_key','reference_period_present','geography_versioned']].all(axis=1)
print(records.to_string(index=False))

source_id   declared_key  unique_key  reference_period_present  geography_versioned  contract_pass
AGR_FRAME     holding_id        True                      True                 True           True
   HMIS_M facility_month        True                      True                False          False
   EMIS_T    school_term        True                      True                 True           True
 SETTLE_G    grid_period        True                      True                 True           True


## 3. Classification and unit controls
Values become comparable only after concepts, units, code lists, and versions are aligned. Never silently map an unknown code to “other”.

In [3]:
code_map=pd.DataFrame({'raw':['MZ','MZ','PRI','URB'],'sector':['Agriculture','Agriculture','Education','Habitat'],'concept':['maize','maize','primary school','urban settlement'],'unit':['tonnes','kilograms','school','grid cell'],'version':['CPC2.1','CPC2.1','ISCED2011','DEGURBA2021']})
code_map['unit_standard']=['kg','kg','school','grid cell']
code_map['factor']=[1000,1,1,1]
assert code_map.version.notna().all()
code_map

## 4. Provenance ledger
Every transformation should have an input, rule, output, actor, time, and version. A provenance hash detects later change; it does not prove the data are true.

In [4]:
import hashlib,datetime
ledger=[]
def record_step(inp,rule,out,actor='controlled-notebook'):
 payload=f'{inp}|{rule}|{out}'.encode(); ledger.append({'input':inp,'rule':rule,'output':out,'actor':actor,'sha256':hashlib.sha256(payload).hexdigest()})
record_step('AGR_FRAME.raw','normalize tonnes to kg','AGR_FRAME.standard')
record_step('EMIS_T.raw','validate ISCED code','EMIS_T.validated')
print(pd.DataFrame(ledger).to_string(index=False))

        input                   rule             output               actor                                                           sha256
AGR_FRAME.raw normalize tonnes to kg AGR_FRAME.standard controlled-notebook 4cb78b58082d4da072073fd92fff53b8612cfe407c60a3d2a0f19f63684e1c97
   EMIS_T.raw    validate ISCED code   EMIS_T.validated controlled-notebook 78c62568fb49dff7914c9ae62db5b383da99fc3667edeb71a8deb9778df496eb


## 5. Linkage is a governed operation
Joining datasets can create new sensitivity and new error. Specify join cardinality, permitted fields, match review, unresolved cases, and whether the linked output may persist.

In [5]:
left=pd.DataFrame({'school_id':['S1','S2','S3'],'attendance':[.91,.74,.86]})
right=pd.DataFrame({'school_id':['S1','S2','S3'],'district':['D1','D1','D2']})
linked=left.merge(right,on='school_id',how='left',validate='one_to_one',indicator=True)
assert (linked._merge=='both').all()
print(linked.to_string(index=False))

school_id  attendance district _merge
       S1        0.91       D1   both
       S2        0.74       D1   both
       S3        0.86       D2   both


## 6. Data minimization and retention
Collect the least detail needed for the decision. Retention requires a purpose, duration, custodian, deletion/archive trigger, and audit record.

In [6]:
fields=pd.DataFrame({'field':['school_id','term','attendance_rate','learner_name','phone'],'needed_for_aggregate_decision':[True,True,True,False,False]})
minimum=fields[fields.needed_for_aggregate_decision].field.tolist()
print('MINIMUM_FIELDS',minimum)
assert 'learner_name' not in minimum

MINIMUM_FIELDS ['school_id', 'term', 'attendance_rate']


## Exercises
1. Draft a source contract for rainfall stations. 2. State the key and grain for monthly facility reporting. 3. Explain why a successful join may still be invalid. 4. Define a retention schedule for a linked analytical table.

## Exact solutions
1. Include station entity, identifier, coordinates/version, variable/unit, observation time, authority, calibration, update cycle, access, retention, and quality owner. 2. A candidate key is facility identifier + reporting month, subject to the declared service/disaggregation grain. 3. Keys can match while concepts, periods, populations, authority, or quality are incompatible. 4. State purpose, lawful basis, approved users, active duration, review date, deletion/archive rule, and evidence of disposal.

In [7]:
assert len(contracts)==4 and all(c.authority for c in contracts)
assert records.contract_pass.sum()==3
print('V7_0_N02_COMPLETE_EXECUTION_PASS')

V7_0_N02_COMPLETE_EXECUTION_PASS
